# 03 — Anomaly Detection Modeling
**Project:** Behavioral Anomaly Detection in Human Chess

This notebook trains and compares three anomaly detection models:
1. **Isolation Forest** — tree-based, fast, interpretable
2. **One-Class SVM** — kernel-based boundary learning
3. **Autoencoder** — neural reconstruction error

All models share the same feature matrix and are compared via a consistent scoring interface.

In [ ]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "src").is_dir():
    _root = (_root / "..").resolve()
sys.path.insert(0, str(_root))

RESULTS_DIR = _root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_and_prepare
from src.features import aggregate_player_stats, add_engineered_features, get_feature_matrix
from src.models import IsolationForestDetector, OneClassSVMDetector, AutoencoderDetector, run_all_models
from src.validation import (
    inject_synthetic_anomalies, evaluate_injection_recovery,
    test_anomaly_vs_normal, compute_silhouette, compute_davies_bouldin
)

## 1. Prepare Feature Matrix

In [ ]:
game_df, player_df = load_and_prepare()
agg = aggregate_player_stats(player_df)
agg = add_engineered_features(agg)

# Analyze blitz games (most common, most consistent)
X, meta, scaler = get_feature_matrix(agg, use_acpl=False, time_control='blitz')
X_arr = X.values
feature_names = list(X.columns)

print(f'Feature matrix: {X_arr.shape}')
X.describe()

## 2. Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'feature_correlation.png'), dpi=150)
plt.show()

## 3. Run All Models (Ensemble)

In [ ]:
results = run_all_models(X_arr, meta)
print(f'\nAnomaly votes distribution:')
print(results['anomaly_votes'].value_counts())
print(f'\nEnsemble anomalies flagged: {results["ensemble_anomaly"].sum()}')
results.head(10)

## 4. Model Comparison — Anomaly Score Distributions

In [ ]:
score_cols = [c for c in results.columns if c.endswith('_score')]
fig, axes = plt.subplots(1, len(score_cols), figsize=(14, 4))

for ax, col in zip(axes, score_cols):
    model_name = col.replace('_score', '')
    label_col = f'{model_name}_label'
    for label, color, name in [(-1, 'crimson', 'Anomaly'), (1, 'steelblue', 'Normal')]:
        mask = results[label_col] == label
        ax.hist(results.loc[mask, col], bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(model_name)
    ax.set_xlabel('Anomaly Score')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'model_score_distributions.png'), dpi=150)
plt.show()

## 5. Top Flagged Players

In [ ]:
# Show players flagged by all 3 models
consensus = results[results['anomaly_votes'] == 3].sort_values('IsolationForest_score', ascending=False)
print(f'Players flagged by ALL 3 models: {len(consensus)}')
consensus[['player_id', 'avg_rating', 'n_games', 'anomaly_votes']].head(20)

## 6. Synthetic Validation

In [ ]:
# Train fresh IF for injection test
if_model = IsolationForestDetector()
if_model.fit(X_arr)

X_inj, y_inj = inject_synthetic_anomalies(X, n=50, strategy='engine_perfect')
injection_results = evaluate_injection_recovery(if_model, X_inj, y_inj)

print('Synthetic Injection Recovery:')
for k, v in injection_results.items():
    print(f'  {k}: {v}')

## 7. Statistical Divergence Tests

In [ ]:
if_labels = if_model.predict(X_arr)

divergence_df = test_anomaly_vs_normal(
    anomaly_scores=if_model.score(X_arr),
    labels=if_labels,
    feature_matrix=X_arr,
    feature_names=feature_names,
)

print('Statistical tests (anomaly vs. normal):')
divergence_df

In [ ]:
sil = compute_silhouette(X_arr, if_labels)
db = compute_davies_bouldin(X_arr, if_labels)
print(f'Silhouette Score: {sil:.4f}  (higher is better, max=1)')
print(f'Davies-Bouldin Index: {db:.4f}  (lower is better)')

## 8. Model Comparison Summary Table

*(Fill in with your results after running)*

| Model | N Anomalies | Synthetic Recovery (P@k) | Silhouette | Notes |
|---|---|---|---|---|
| Isolation Forest | ? | ? | ? | Fast, interpretable |
| One-Class SVM | ? | ? | ? | Kernel-based boundary |
| Autoencoder | ? | ? | ? | Neural reconstruction |
| Ensemble (2/3 vote) | ? | ? | ? | Most conservative |